# DeepFaceLive LIA portrait-avatar GPU audit
Tests the portrait animator already included in the existing project, using the verified CUDA-12 ONNX package family. No PyTorch, public endpoint, paid GPU, camera or microphone is needed. Select a free Kaggle T4 and Internet. Save & Run All retains complete outputs and logs.

A pass requires executed CUDA compute nodes for the detector, landmarks and portrait generator, actual animated output, no-face hold, calibration reset, and an HTTP/WebRTC round trip. Native generation is 256 x 256. The uploaded portrait supplies the head, hair, visible clothes and background; this does not generate full-body movement or audio-driven lip sync. Only fictional project samples are used for this test.


In [ ]:
# DeepFaceLive's existing LIA portrait animator: actual CUDA and WebRTC audit.
# Uses the existing experiment branch. No public route or paid GPU is started.
from pathlib import Path
import sys, subprocess
repo=Path('/kaggle/working/DeepFaceLive')
branch='experiment/cloud-headless-webrtc'
commit='ec548ecf4e40d1246223536b9368a227ad38b0d1'
fresh=not (repo/'.git').is_dir()
if fresh:
    subprocess.run(['git','clone','--filter=blob:none','--depth','1','--no-checkout','--single-branch','--branch',branch,'https://github.com/johnlilda2-ui/DeepFaceLive.git',str(repo)],check=True,timeout=300)
    patterns=['/scripts/','/xlib/','/modelhub/__init__.py','/modelhub/onnx/__init__.py','/modelhub/onnx/*/*.py','/modelhub/onnx/InsightFace2d106/InsightFace2D106.onnx','/modelhub/onnx/YoloV5Face/YoloV5Face.onnx','/modelhub/onnx/LIA/generator.onnx.part*','/doc/celebs/Rob_Doe/Rob_Doe.png','/doc/celebs/Millie_Park/Millie_Park.png']
    subprocess.run(['git','-C',str(repo),'sparse-checkout','set','--no-cone',*patterns],check=True)
else:
    subprocess.run(['git','-C',str(repo),'sparse-checkout','add','--no-cone','/modelhub/onnx/LIA/generator.onnx.part*'],check=True)
subprocess.run(['git','-C',str(repo),'fetch','--depth','1','origin',commit],check=True,timeout=300)
subprocess.run(['git','-C',str(repo),'checkout','-B',branch,'FETCH_HEAD'],check=True)
env=Path('/kaggle/working/dfl-cuda12-env')
python=str(env/'bin/python')
subprocess.run([sys.executable,str(repo/'scripts/prepare_headless_env.py'),str(env)],check=True)
# Same CUDA-12 package family as the verified face-swap environment; no PyTorch.
packages=['onnxruntime-gpu[cuda,cudnn]==1.22.0','numpy==2.2.6','onnx==1.18.0','opencv-python-headless==4.11.0.86','numexpr==2.10.2','h5py==3.12.1','wrapt==1.17.2','psutil==7.0.0','Pillow==11.3.0','aiohttp==3.13.3','aiortc==1.15.0','av==17.0.0','pyee==13.0.1','nvidia-cublas-cu12==12.9.3.13','nvidia-cuda-runtime-cu12==12.9.79','nvidia-cudnn-cu12==9.28.0.42']
subprocess.run([sys.executable,'-m','pip','--python',python,'install','--no-cache-dir',*packages],check=True,timeout=900)
source=repo/'doc/celebs/Millie_Park/Millie_Park.png'
driver=repo/'doc/celebs/Rob_Doe/Rob_Doe.png'
subprocess.run([python,'-u',str(repo/'scripts/test_portrait_gpu.py'),'--engine','lia','--source-image',str(source),'--driver-image',str(driver),'--second-driver-image',str(source),'--output-dir','/kaggle/working/dfl-lia-portrait-audit'],cwd=repo,check=True,timeout=180)
subprocess.run([python,'-u',str(repo/'scripts/test_signaling_gpu_roundtrip.py'),'--portrait-engine','lia','--source-image',str(source),'--target-image',str(driver),'--output-dir','/kaggle/working/dfl-lia-portrait-webrtc'],cwd=repo,check=True,timeout=180)
print('LIA_PORTRAIT_GPU_AND_WEBRTC_PASSED')
from IPython.display import display, Image
reports=sorted(Path('/kaggle/working/dfl-lia-portrait-audit').glob('run-*/portrait-report.json'),key=lambda p:p.stat().st_mtime)
print(reports[-1].read_text())
display(Image(filename=str(reports[-1].parent/'avatar-0.png')))
display(Image(filename=str(reports[-1].parent/'avatar-5.png')))
